# negative-back — worked example 3: negative_back composed three times

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `negative-back`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Single-parent backward functions compose by ordinary function composition. Threading `negative_back` through an odd number of negations leaves a net sign flip, while an even number cancels. Three negations give `y = -(-(-x))`, so the leaf gradient is `-grad_out`.

## Worked solution

We define `negative_back` and a helper `chain_triple_negate(grad_out, x)` that runs the reverse pass over `y = -(-(-x))`. The forward builds `u = -x`, `v = -u`, `y = -v`. Going backward, each stage flips the sign: `g_v = -grad_out`, `g_u = -g_v = grad_out`, `g_x = -g_u = -grad_out`. Three flips net to one flip. We seed, draw a `(3,)` vector and a random `grad_out`, run the chain, and confirm the result equals `-grad_out` exactly (no tolerance needed because the operation is exact sign flipping). We print the comparison.

In [ ]:
import torch as t

t.manual_seed(2)

def negative_back(grad_out, out, x):
    return -grad_out

def chain_triple_negate(grad_out, x):
    u = -x
    v = -u
    y = -v
    g_v = negative_back(grad_out, y, v)
    g_u = negative_back(g_v, v, u)
    g_x = negative_back(g_u, u, x)
    return g_x

x = t.randn(3)
grad_out = t.randn(3)
g_x = chain_triple_negate(grad_out, x)
print(g_x)
print('equals -grad_out:', t.equal(g_x, -grad_out))